In [1]:
import numpy as np 
import polars as pl 
import polars.selectors as cs
import pandas as pd 
import utils

In [2]:
# import data
train = pl.read_parquet("../data/cleaned/application_train.parquet")
test  = pl.read_parquet("../data/cleaned/application_test.parquet")
buro  = pl.read_parquet("../data/cleaned/bureau.parquet")
bbal  = pl.read_parquet("../data/cleaned/bureau_balance.parquet")
prev  = pl.read_parquet("../data/cleaned/previous_application.parquet")
card  = pl.read_parquet("../data/cleaned/credit_card_balance.parquet")
poca  = pl.read_parquet("../data/cleaned/POS_CASH_balance.parquet")
inst  = pl.read_parquet("../data/cleaned/installments_payments.parquet")

In [3]:
# ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [4]:
# garbage collection
import gc
gc.enable()

In [5]:
# check dimensions
print("Application:", train.shape, test.shape)
print("Buro:", buro.shape)
print("Bbal:", bbal.shape)
print("Prev:", prev.shape)
print("Card:", card.shape)
print("Poca:", poca.shape)
print("Inst:", inst.shape)

Application: (307511, 122) (48744, 121)
Buro: (1716428, 17)
Bbal: (27299925, 3)
Prev: (1670214, 37)
Card: (3840312, 23)
Poca: (10001358, 8)
Inst: (13605401, 8)


In [6]:
# extract target
y = train.select(["SK_ID_CURR", "TARGET"])
train = train.drop("TARGET")

In [7]:
# concatenate application data
appl = pl.concat([train, test])
del train, test

In [8]:
import polars as pl
import utils

# list of documents
doc_vars = ["FLAG_DOCUMENT_2",  "FLAG_DOCUMENT_3",  "FLAG_DOCUMENT_4",  "FLAG_DOCUMENT_5",  "FLAG_DOCUMENT_6",
            "FLAG_DOCUMENT_7",  "FLAG_DOCUMENT_8",  "FLAG_DOCUMENT_9",  "FLAG_DOCUMENT_10", "FLAG_DOCUMENT_11",
            "FLAG_DOCUMENT_12", "FLAG_DOCUMENT_13", "FLAG_DOCUMENT_14", "FLAG_DOCUMENT_15", "FLAG_DOCUMENT_16",
            "FLAG_DOCUMENT_17", "FLAG_DOCUMENT_18", "FLAG_DOCUMENT_19", "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21"]

# 1. Feature Engineering (computed in parallel)
appl = appl.with_columns(
    # income ratios
    (pl.col("AMT_CREDIT") / pl.col("AMT_INCOME_TOTAL")).alias("CREDIT_BY_INCOME"),
    (pl.col("AMT_ANNUITY") / pl.col("AMT_INCOME_TOTAL")).alias("ANNUITY_BY_INCOME"),
    (pl.col("AMT_GOODS_PRICE") / pl.col("AMT_INCOME_TOTAL")).alias("GOODS_PRICE_BY_INCOME"),
    (pl.col("AMT_INCOME_TOTAL") / pl.col("CNT_FAM_MEMBERS")).alias("INCOME_PER_PERSON"),
    
    # career ratio (replaces negatives with None)
    pl.when((pl.col("DAYS_EMPLOYED") / pl.col("DAYS_BIRTH")) < 0)
      .then(None)
      .otherwise(pl.col("DAYS_EMPLOYED") / pl.col("DAYS_BIRTH"))
      .alias("PERCENT_WORKED"),
      
    # number of adults and children ratio
    (pl.col("CNT_FAM_MEMBERS") - pl.col("CNT_CHILDREN")).alias("CNT_ADULTS"),
    (pl.col("CNT_CHILDREN") / pl.col("CNT_FAM_MEMBERS")).alias("CHILDREN_RATIO"),
    
    # overall payments
    (pl.col("AMT_CREDIT") / pl.col("AMT_ANNUITY")).alias("ANNUITY LENGTH"),
    
    # external sources
    pl.mean_horizontal("EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3").alias("EXT_SOURCE_MEAN"),
    # sum_horizontal on is_not_null() perfectly mimics the "3 - sum(is_null)" logic natively
    pl.sum_horizontal(
        pl.col("EXT_SOURCE_1").is_not_null(),
        pl.col("EXT_SOURCE_2").is_not_null(),
        pl.col("EXT_SOURCE_3").is_not_null()
    ).alias("NUM_EXT_SOURCES"),
    
    # number of documents
    pl.sum_horizontal(doc_vars).alias("NUM_DOCUMENTS"),
    
    # application date (Weekend / Working day)
    pl.when(pl.col("WEEKDAY_APPR_PROCESS_START").is_in(["SATURDAY", "SUNDAY"]))
      .then(pl.lit("Weekend"))
      .otherwise(pl.lit("Working day"))
      .alias("DAY_APPR_PROCESS_START"),
      
    # age ratios
    (pl.col("OWN_CAR_AGE") / pl.col("DAYS_BIRTH")).alias("OWN_CAR_AGE_RATIO"),
    (pl.col("DAYS_ID_PUBLISH") / pl.col("DAYS_BIRTH")).alias("DAYS_ID_PUBLISHED_RATIO"),
    (pl.col("DAYS_REGISTRATION") / pl.col("DAYS_BIRTH")).alias("DAYS_REGISTRATION_RATIO"),
    (pl.col("DAYS_LAST_PHONE_CHANGE") / pl.col("DAYS_BIRTH")).alias("DAYS_LAST_PHONE_CHANGE_RATIO")
)

# 2. Apply Custom Functions (These now work correctly with the Polars DataFrame)
log_vars = ["AMT_CREDIT", "AMT_INCOME_TOTAL", "AMT_GOODS_PRICE", "AMT_ANNUITY"]
appl = utils.create_logarithms(appl, log_vars, replace=True)

day_vars = ["DAYS_BIRTH", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_EMPLOYED", "DAYS_LAST_PHONE_CHANGE"]
appl = utils.convert_days(appl, day_vars, t=30, rounding=True, replace=True)

# 3. Drop unused features
drops = ['APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 
         'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI',
         'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI','YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI',
         'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'COMMONAREA_MODE','ELEVATORS_MODE', 'ENTRANCES_MODE', 
         'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 
         'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'TOTALAREA_MODE',  'YEARS_BEGINEXPLUATATION_MODE']

appl = appl.drop(drops)


In [9]:
rename_mapping = {col: f"app_{col}" for col in appl.columns if col != "SK_ID_CURR"}
appl.rename(rename_mapping)

SK_ID_CURR,app_NAME_CONTRACT_TYPE,app_CODE_GENDER,app_FLAG_OWN_CAR,app_FLAG_OWN_REALTY,app_CNT_CHILDREN,app_AMT_INCOME_TOTAL,app_AMT_CREDIT,app_AMT_ANNUITY,app_AMT_GOODS_PRICE,app_NAME_TYPE_SUITE,app_NAME_INCOME_TYPE,app_NAME_EDUCATION_TYPE,app_NAME_FAMILY_STATUS,app_NAME_HOUSING_TYPE,app_REGION_POPULATION_RELATIVE,app_DAYS_BIRTH,app_DAYS_EMPLOYED,app_DAYS_REGISTRATION,app_DAYS_ID_PUBLISH,app_OWN_CAR_AGE,app_FLAG_MOBIL,app_FLAG_EMP_PHONE,app_FLAG_WORK_PHONE,app_FLAG_CONT_MOBILE,app_FLAG_PHONE,app_FLAG_EMAIL,app_OCCUPATION_TYPE,app_CNT_FAM_MEMBERS,app_REGION_RATING_CLIENT,app_REGION_RATING_CLIENT_W_CITY,app_WEEKDAY_APPR_PROCESS_START,app_HOUR_APPR_PROCESS_START,app_REG_REGION_NOT_LIVE_REGION,app_REG_REGION_NOT_WORK_REGION,app_LIVE_REGION_NOT_WORK_REGION,app_REG_CITY_NOT_LIVE_CITY,…,app_FLAG_DOCUMENT_7,app_FLAG_DOCUMENT_8,app_FLAG_DOCUMENT_9,app_FLAG_DOCUMENT_10,app_FLAG_DOCUMENT_11,app_FLAG_DOCUMENT_12,app_FLAG_DOCUMENT_13,app_FLAG_DOCUMENT_14,app_FLAG_DOCUMENT_15,app_FLAG_DOCUMENT_16,app_FLAG_DOCUMENT_17,app_FLAG_DOCUMENT_18,app_FLAG_DOCUMENT_19,app_FLAG_DOCUMENT_20,app_FLAG_DOCUMENT_21,app_AMT_REQ_CREDIT_BUREAU_HOUR,app_AMT_REQ_CREDIT_BUREAU_DAY,app_AMT_REQ_CREDIT_BUREAU_WEEK,app_AMT_REQ_CREDIT_BUREAU_MON,app_AMT_REQ_CREDIT_BUREAU_QRT,app_AMT_REQ_CREDIT_BUREAU_YEAR,app_CREDIT_BY_INCOME,app_ANNUITY_BY_INCOME,app_GOODS_PRICE_BY_INCOME,app_INCOME_PER_PERSON,app_PERCENT_WORKED,app_CNT_ADULTS,app_CHILDREN_RATIO,app_ANNUITY LENGTH,app_EXT_SOURCE_MEAN,app_NUM_EXT_SOURCES,app_NUM_DOCUMENTS,app_DAY_APPR_PROCESS_START,app_OWN_CAR_AGE_RATIO,app_DAYS_ID_PUBLISHED_RATIO,app_DAYS_REGISTRATION_RATIO,app_DAYS_LAST_PHONE_CHANGE_RATIO
i32,str,str,str,str,i32,f32,f32,f32,f32,str,str,str,str,str,f32,f64,f64,f32,f64,f32,i32,i32,i32,i32,i32,i32,str,f32,i32,i32,str,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f32,f32,u32,i32,str,f64,f64,f64,f64
100002,"""Cash loans""","""M""","""N""","""Y""",0,12.218501,12.915583,10.11462,12.768545,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.018801,315.0,21.0,122.0,71.0,null,1,1,0,1,1,0,"""Laborers""",1.0,2,2,"""WEDNESDAY""",10,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,2.007889,0.121978,1.733333,202500.0,0.067329,1.0,0.0,16.461103,0.161787,3,1,"""Working day""",null,0.224078,0.385583,0.11986
100003,"""Cash loans""","""F""","""N""","""N""",0,12.506182,14.072865,10.482893,13.937287,"""Family""","""State servant""","""Higher education""","""Married""","""House / apartment""",0.003541,559.0,40.0,40.0,10.0,null,1,1,0,1,1,0,"""Core staff""",2.0,1,1,"""MONDAY""",11,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,4.79075,0.132217,4.183333,135000.0,0.070862,2.0,0.0,36.234085,0.466757,2,1,"""Working day""",null,0.017358,0.070743,0.049389
100004,"""Revolving loans""","""M""","""Y""","""Y""",0,11.119899,11.813039,8.817447,11.813039,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.010032,635.0,8.0,142.0,84.0,26.0,1,1,1,1,1,0,"""Laborers""",1.0,2,2,"""MONDAY""",9,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.1,2.0,67500.0,0.011814,1.0,0.0,20.0,0.642739,2,0,"""Working day""",-0.001365,0.132889,0.223669,0.042791
100006,"""Cash loans""","""F""","""N""","""Y""",0,11.813039,12.652947,10.298482,12.601492,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Civil marriage""","""House / apartment""",0.008019,634.0,101.0,328.0,81.0,null,1,1,0,1,0,0,"""Laborers""",2.0,2,2,"""WEDNESDAY""",17,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,null,null,null,null,null,null,2.316167,0.2199,2.2,67500.0,0.159905,2.0,0.0,10.532818,0.650442,1,1,"""Working day""",null,0.128229,0.51739,0.032465
100007,"""Cash loans""","""M""","""N""","""Y""",0,11.707679,13.148034,9.992712,13.148034,"""Unaccompanied""","""Working""","""Secondary / secondary s

In [10]:
# check data
appl.head()

SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,…,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,CREDIT_BY_INCOME,ANNUITY_BY_INCOME,GOODS_PRICE_BY_INCOME,INCOME_PER_PERSON,PERCENT_WORKED,CNT_ADULTS,CHILDREN_RATIO,ANNUITY LENGTH,EXT_SOURCE_MEAN,NUM_EXT_SOURCES,NUM_DOCUMENTS,DAY_APPR_PROCESS_START,OWN_CAR_AGE_RATIO,DAYS_ID_PUBLISHED_RATIO,DAYS_REGISTRATION_RATIO,DAYS_LAST_PHONE_CHANGE_RATIO
i32,str,str,str,str,i32,f32,f32,f32,f32,str,str,str,str,str,f32,f64,f64,f32,f64,f32,i32,i32,i32,i32,i32,i32,str,f32,i32,i32,str,i32,i32,i32,i32,i32,…,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,f32,f32,u32,i32,str,f64,f64,f64,f64
100002,"""Cash loans""","""M""","""N""","""Y""",0,12.218501,12.915583,10.11462,12.768545,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.018801,315.0,21.0,122.0,71.0,null,1,1,0,1,1,0,"""Laborers""",1.0,2,2,"""WEDNESDAY""",10,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,2.007889,0.121978,1.733333,202500.0,0.067329,1.0,0.0,16.461103,0.161787,3,1,"""Working day""",null,0.224078,0.385583,0.11986
100003,"""Cash loans""","""F""","""N""","""N""",0,12.506182,14.072865,10.482893,13.937287,"""Family""","""State servant""","""Higher education""","""Married""","""House / apartment""",0.003541,559.0,40.0,40.0,10.0,null,1,1,0,1,1,0,"""Core staff""",2.0,1,1,"""MONDAY""",11,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,4.79075,0.132217,4.183333,135000.0,0.070862,2.0,0.0,36.234085,0.466757,2,1,"""Working day""",null,0.017358,0.070743,0.049389
100004,"""Revolving loans""","""M""","""Y""","""Y""",0,11.119899,11.813039,8.817447,11.813039,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.010032,635.0,8.0,142.0,84.0,26.0,1,1,1,1,1,0,"""Laborers""",1.0,2,2,"""MONDAY""",9,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.1,2.0,67500.0,0.011814,1.0,0.0,20.0,0.642739,2,0,"""Working day""",-0.001365,0.132889,0.223669,0.042791
100006,"""Cash loans""","""F""","""N""","""Y""",0,11.813039,12.652947,10.298482,12.601492,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Civil marriage""","""House / apartment""",0.008019,634.0,101.0,328.0,81.0,null,1,1,0,1,0,0,"""Laborers""",2.0,2,2,"""WEDNESDAY""",17,0,0,0,0,…,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,null,null,null,null,null,null,2.316167,0.2199,2.2,67500.0,0.159905,2.0,0.0,10.532818,0.650442,1,1,"""Working day""",null,0.128229,0.51739,0.032465
100007,"""Cash loans""","""M""","""N""","""Y""",0,11.707679,13.148034,9.992712,13.148034,"""Unaccompanied""","""Working""","""Secondary / secondary special""","""Single / not married""","""House / apartment""",0.028663,664.0,101.0,144.0,115.0,null,1,1,0,1,0,0,"""Core staff""",1.0,2,2,"""THURSDAY""",11,0,0,0,0,…,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,4.222222,0.179963,4.222222,121500.0,0.152418,1.0,0.0,23.461618,0.322738,1,

In [11]:
# count missings
nas = utils.count_missings(appl)
nas.head()

Feature,Total,Percent
str,i64,f64
"""COMMONAREA_AVG""",248360,69.714109
"""NONLIVINGAPARTMENTS_AVG""",246861,69.293343
"""FONDKAPREMONT_MODE""",243092,68.235393
"""LIVINGAPARTMENTS_AVG""",242979,68.203674
"""FLOORSMIN_AVG""",241108,67.678489


In [12]:
# check bbal data
bbal.head()

SK_ID_BUREAU,MONTHS_BALANCE,STATUS
i32,i32,str
5715448,0,"""C"""
5715448,-1,"""C"""
5715448,-2,"""C"""
5715448,-3,"""C"""
5715448,-4,"""C"""


In [13]:
loan_score = (
    bbal
    .with_columns(
            pl.col("STATUS").replace({"X": None, "1": 1.0, "2": 2.0, "3": 3.0, "4": 4.0, "5": 5.0}, default=0.0
    ).alias("NUM_STATUS")
    )
    .with_columns(
        (pl.col("NUM_STATUS")/(pl.col("MONTHS_BALANCE").abs()+1)).alias("LOAN_SCORE")
    )
    .group_by("SK_ID_BUREAU")
    .agg(pl.col("LOAN_SCORE").sum())
)

bbal = bbal.to_dummies("STATUS")

In [14]:
# count missings
nas = utils.count_missings(bbal)
nas.head()

Feature,Total,Percent
str,i64,f64


In [15]:
agg_bbal = (
    bbal
    .group_by("SK_ID_BUREAU")
    .agg(
        pl.col("MONTHS_BALANCE").count().alias("MONTH_COUNT"),
        cs.starts_with("STATUS_").mean()
    )
    .join(loan_score,on= "SK_ID_BUREAU",how = "left")
)

In [16]:
# count missings
nas = utils.count_missings(agg_bbal)
nas.head()

Feature,Total,Percent
str,i64,f64


In [17]:
# check data
agg_bbal.head()

SK_ID_BUREAU,MONTH_COUNT,STATUS_0,STATUS_1,STATUS_2,STATUS_3,STATUS_4,STATUS_5,STATUS_C,STATUS_X,LOAN_SCORE
i32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64
5863477,26,0.115385,0.0,0.0,0.0,0.0,0.0,0.0,0.884615,0.0
5439053,46,0.0,0.0,0.0,0.0,0.0,0.0,0.217391,0.782609,0.0
5539203,12,0.333333,0.0,0.0,0.0,0.0,0.0,0.666667,0.0,0.0
6290149,89,0.011236,0.0,0.0,0.0,0.0,0.0,0.898876,0.089888,0.0
6388840,6,0.833333,0.0,0.0,0.0,0.0,0.0,0.0,0.166667,0.0


In [18]:
# clear memory
del bbal

In [19]:
# check buro data
buro.head()

SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
i32,i32,str,str,i32,i32,f32,f32,f32,i32,f32,f32,f32,f32,str,i32,f32
215354,5714462,"""Closed""","""currency 1""",-497,0,-153.0,-153.0,null,0,91323.0,0.0,null,0.0,"""Consumer credit""",-131,null
215354,5714463,"""Active""","""currency 1""",-208,0,1075.0,null,null,0,225000.0,171342.0,null,0.0,"""Credit card""",-20,null
215354,5714464,"""Active""","""currency 1""",-203,0,528.0,null,null,0,464323.5,null,null,0.0,"""Consumer credit""",-16,null
215354,5714465,"""Active""","""currency 1""",-203,0,null,null,null,0,90000.0,null,null,0.0,"""Credit card""",-16,null
215354,5714466,"""Active""","""currency 1""",-629,0,1197.0,null,77674.5,0,2.7e6,null,null,0.0,"""Consumer credit""",-21,null


In [20]:
buro = buro.join(agg_bbal, how = "left", on = "SK_ID_BUREAU")

In [21]:
# Total bureau loans per applicant
buro = buro.with_columns(
    pl.len().over("SK_ID_CURR").alias("CNT_BURO_LOANS")
)


buro = buro.with_columns(
    (pl.col("AMT_CREDIT_SUM_OVERDUE") / pl.col("AMT_ANNUITY")).alias("AMT_SUM_OVERDUE_RATIO_1"),
    (pl.col("AMT_CREDIT_SUM_OVERDUE") / pl.col("AMT_CREDIT_SUM")).alias("AMT_SUM_OVERDUE_RATIO_2"),
    (pl.col("AMT_CREDIT_MAX_OVERDUE") / pl.col("AMT_ANNUITY")).alias("AMT_MAX_OVERDUE_RATIO_1"),
    (pl.col("AMT_CREDIT_MAX_OVERDUE") / pl.col("AMT_CREDIT_SUM")).alias("AMT_MAX_OVERDUE_RATIO_2"),
    (pl.col("AMT_CREDIT_SUM_DEBT") / pl.col("AMT_CREDIT_SUM")).alias("AMT_SUM_DEBT_RATIO_1"),
    (pl.col("AMT_CREDIT_SUM_DEBT") / pl.col("AMT_CREDIT_SUM_LIMIT")).alias("AMT_SUM_DEBT_RATIO_2"),
)

log_vars = ["AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_OVERDUE", "AMT_ANNUITY"]
buro = utils.create_logarithms(buro, log_vars, replace=True)

# Convert Days
day_vars = ["DAYS_CREDIT", "CREDIT_DAY_OVERDUE", "DAYS_CREDIT_ENDDATE", "DAYS_ENDDATE_FACT", "DAYS_CREDIT_UPDATE"]
buro = utils.convert_days(buro, day_vars, t=1, rounding=False, replace=True)

# 5. Remaining engineered features
buro = buro.with_columns(
    # Recency-weighted loan score
    (pl.col("LOAN_SCORE") / (pl.col("DAYS_CREDIT") / 12)).alias("WEIGHTED_LOAN_SCORE"),
    
    (pl.col("DAYS_ENDDATE_FACT") - pl.col("DAYS_CREDIT_ENDDATE")).alias("DAYS_END_DIFF_1"),
    (pl.col("DAYS_CREDIT_UPDATE") - pl.col("DAYS_CREDIT_ENDDATE")).alias("DAYS_END_DIFF_2"),
    (pl.col("DAYS_CREDIT_ENDDATE") - pl.col("DAYS_CREDIT")).alias("DAYS_DURATION_1"),
    (pl.col("DAYS_ENDDATE_FACT") - pl.col("DAYS_CREDIT")).alias("DAYS_DURATION_2"),
    
    (pl.col("CREDIT_ACTIVE") == "Active").sum().over("SK_ID_CURR").alias("CNT_BURO_ACTIVE"),
    (pl.col("CREDIT_ACTIVE") == "Closed").sum().over("SK_ID_CURR").alias("CNT_BURO_CLOSED"),
    (pl.col("CREDIT_ACTIVE") == "Bad debt").sum().over("SK_ID_CURR").alias("CNT_BURO_BAD")
)


In [22]:
buro = buro.to_dummies(cs.string(),drop_first=True)

In [23]:
# count missings
nas = utils.count_missings(buro)
nas.head()

Feature,Total,Percent
str,i64,f64
"""AMT_MAX_OVERDUE_RATIO_1""",1598872,93.151125
"""AMT_ANNUITY""",1226791,71.47349
"""AMT_SUM_OVERDUE_RATIO_1""",1226791,71.47349
"""AMT_CREDIT_MAX_OVERDUE""",1124488,65.513264
"""AMT_MAX_OVERDUE_RATIO_2""",1124488,65.513264


In [24]:
cnt_buro = (
    buro
    .group_by("SK_ID_CURR")
    .agg(
        pl.col("SK_ID_BUREAU").count().alias("buro_BURO_COUNT")
    )
)
buro = buro.drop("SK_ID_BUREAU")

agg_buro = utils.aggregate_data(buro,id_var= "SK_ID_CURR",label = "buro")

agg_buro = agg_buro.join(cnt_buro,on="SK_ID_CURR",how="left")

agg_buro = agg_buro.drop([
    "buro_WEIGHTED_LOAN_SCORE_std", 
    "buro_WEIGHTED_LOAN_SCORE_min", 
    "buro_WEIGHTED_LOAN_SCORE_max"
])

- Preparing the dataset...
- Extracted 0 factors and 57 numerics...
- Aggregating numeric features...
- Final dimensions: (305811, 229)


In [25]:
# count missings
nas = utils.count_missings(agg_buro)
nas.head()

Feature,Total,Percent
str,i64,f64
"""buro_AMT_MAX_OVERDUE_RATIO_1_s…",277289,90.673324
"""buro_AMT_MAX_OVERDUE_RATIO_1_m…",242976,79.452995
"""buro_AMT_MAX_OVERDUE_RATIO_1_m…",242976,79.452995
"""buro_AMT_MAX_OVERDUE_RATIO_1_m…",242976,79.452995
"""buro_AMT_ANNUITY_std""",213412,69.785587


In [29]:
# check data
print(agg_buro.shape)
agg_buro.head()


(305811, 227)


SK_ID_CURR,buro_CREDIT_ACTIVE_Active_mean,buro_CREDIT_ACTIVE_Bad debt_mean,buro_CREDIT_ACTIVE_Sold_mean,buro_CREDIT_CURRENCY_currency 2_mean,buro_CREDIT_CURRENCY_currency 3_mean,buro_CREDIT_CURRENCY_currency 4_mean,buro_DAYS_CREDIT_mean,buro_CREDIT_DAY_OVERDUE_mean,buro_DAYS_CREDIT_ENDDATE_mean,buro_DAYS_ENDDATE_FACT_mean,buro_AMT_CREDIT_MAX_OVERDUE_mean,buro_CNT_CREDIT_PROLONG_mean,buro_AMT_CREDIT_SUM_mean,buro_AMT_CREDIT_SUM_DEBT_mean,buro_AMT_CREDIT_SUM_LIMIT_mean,buro_AMT_CREDIT_SUM_OVERDUE_mean,buro_CREDIT_TYPE_Another type of loan_mean,buro_CREDIT_TYPE_Car loan_mean,buro_CREDIT_TYPE_Cash loan (non-earmarked)_mean,buro_CREDIT_TYPE_Credit card_mean,buro_CREDIT_TYPE_Interbank credit_mean,buro_CREDIT_TYPE_Loan for business development_mean,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_mean,buro_CREDIT_TYPE_Loan for the purchase of equipment_mean,buro_CREDIT_TYPE_Loan for working capital replenishment_mean,buro_CREDIT_TYPE_Microloan_mean,buro_CREDIT_TYPE_Mobile operator loan_mean,buro_CREDIT_TYPE_Mortgage_mean,buro_CREDIT_TYPE_Real estate loan_mean,buro_CREDIT_TYPE_Unknown type of loan_mean,buro_DAYS_CREDIT_UPDATE_mean,buro_AMT_ANNUITY_mean,buro_MONTH_COUNT_mean,buro_STATUS_0_mean,buro_STATUS_1_mean,buro_STATUS_2_mean,…,buro_CREDIT_TYPE_Interbank credit_max,buro_CREDIT_TYPE_Loan for business development_max,buro_CREDIT_TYPE_Loan for purchase of shares (margin lending)_max,buro_CREDIT_TYPE_Loan for the purchase of equipment_max,buro_CREDIT_TYPE_Loan for working capital replenishment_max,buro_CREDIT_TYPE_Microloan_max,buro_CREDIT_TYPE_Mobile operator loan_max,buro_CREDIT_TYPE_Mortgage_max,buro_CREDIT_TYPE_Real estate loan_max,buro_CREDIT_TYPE_Unknown type of loan_max,buro_DAYS_CREDIT_UPDATE_max,buro_AMT_ANNUITY_max,buro_MONTH_COUNT_max,buro_STATUS_0_max,buro_STATUS_1_max,buro_STATUS_2_max,buro_STATUS_3_max,buro_STATUS_4_max,buro_STATUS_5_max,buro_STATUS_C_max,buro_STATUS_X_max,buro_LOAN_SCORE_max,buro_CNT_BURO_LOANS_max,buro_AMT_SUM_OVERDUE_RATIO_1_max,buro_AMT_SUM_OVERDUE_RATIO_2_max,buro_AMT_MAX_OVERDUE_RATIO_1_max,buro_AMT_MAX_OVERDUE_RATIO_2_max,buro_AMT_SUM_DEBT_RATIO_1_max,buro_AMT_SUM_DEBT_RATIO_2_max,buro_DAYS_END_DIFF_1_max,buro_DAYS_END_DIFF_2_max,buro_DAYS_DURATION_1_max,buro_DAYS_DURATION_2_max,buro_CNT_BURO_ACTIVE_max,buro_CNT_BURO_CLOSED_max,buro_CNT_BURO_BAD_max,buro_BURO_COUNT
i32,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f64,f32,f32,f32,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f64,f64,f64,f64,…,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,f64,f32,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,f32,f32,f32,f32,f32,f32,f32,f64,f64,f64,u32,u32,u32,u32
370592,0.4,0.0,0.0,0.0,0.0,0.0,1413.2,0.0,1772.0,1770.0,0.0,0.0,9.983135,2.632029,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,943.0,8.677248,46.8,0.410312,0.0,0.0,…,0,0,0,0,0,0,0,0,0,0,2272.0,8.912744,84,1.0,0.0,0.0,0.0,0.0,0.0,0.97619,0.411765,0.0,5,0.0,0.0,0.0,0.0,0.87933,NaN,93.0,93.0,-91.0,-63.0,2,3,0,5
391695,0.0,0.0,0.0,0.0,0.0,0.0,2270.0,0.0,444.0,430.0,null,0.0,12.100718,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,391.0,null,null,null,null,null,…,0,0,0,0,0,0,0,0,0,0,391.0,null,null,null,null,null,null,null,null,null,null,null,1,null,0.0,null,null,0.0,NaN,-14.0,-53.0,-1826.0,-1840.0,0,1,0,1
138435,0.571429,0.0,0.0,0.0,0.0,0.0,820.142857,0.0,405.5,458.333344,null,0.0,11.959614,4.500415,0.0,0.0,0.0,0.0,0.0,0.142857,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,249.714286,2.127156,21.428571,0.48567,0.0,0.0,…,0,0,0,0,0,0,0,0,0,0,856.0,7.445044,35,0.916667,0.0,0.0,0.0,0.0,0.0,0.857143,0.529412,0.0,7,0.0,0.0,null,null,0.872419,inf,212.0,211.0,-365.0,-153.0,4,3,0,7
179331,0.214286,0.0,0.0,0.0,0.0,0.0,1117.0,0.0,1647.0,636.272705,1897.275024,0.0,10.856487,1.253323,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,350.0,0.0,26.142857,0.469388,0.0,0.0,…,0,0,0,0,0,0,0,0,0,0,2150.0,0.0,70,1.0,0.0,0.0,0.0,0.0,0.0,0.842857,1.0,0.0,14,NaN,0.0,inf,0.019984,1.0,NaN,36.0,0.0,-304.0,-216.0,3,11,0,14
44

In [30]:
del buro